# نظام RAG التقليدي المتكامل باستخدام ChromaDB (الأجزاء 1، 2، 3)
### Building Complete Traditional RAG Pipeline with ChromaDB & LangChain

دليل عملي شامل يغطي **المحاضرات 25 و 26 و 27**:
1. **المحاضرة 25**: قراءة المستندات وتجزئتها (`RecursiveCharacterTextSplitter`).
2. **المحاضرة 26**: توليد التضمينات (`HuggingFaceEmbeddings`) وحفظها في `Chroma` على القرص.
3. **المحاضرة 27**: تحويل الفهرس إلى `Retriever`، تصميم الـ `Prompt`، وبناء سلسلة `create_retrieval_chain` مع الـ LLM (`ChatGroq`).

In [ ]:
import os
import shutil
from dotenv import load_dotenv

# load environment variables
load_dotenv()

# langchain imports
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents.stuff import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

print("All libraries imported successfully!")

---
## 1. تجهيز وقراءة المستندات النصية (Ingestion)

In [ ]:
# sample documents
docs = {
    "document_0.txt": (
        "Machine learning is a subset of artificial intelligence that learns from data without explicit programming. "
        "Supervised learning uses labeled datasets with algorithms like regression, decision trees, and support vector machines. "
        "Unsupervised learning finds hidden patterns in unlabeled data using clustering techniques."
    ),
    "document_1.txt": (
        "Deep learning is a specialized branch of machine learning inspired by artificial neural networks with multiple layers. "
        "Convolutional Neural Networks (CNNs) are best used for computer vision and image processing tasks. "
        "Recurrent Neural Networks (RNNs) and LSTMs excel at sequential and time-series data. Training relies on backpropagation."
    ),
    "document_2.txt": (
        "Natural language processing (NLP) enables machines to understand, interpret, and generate human language. "
        "The Transformer architecture revolutionized NLP in 2017 with self-attention mechanisms, replacing older recurrence. "
        "Pre-trained models like BERT and GPT set state-of-the-art benchmarks in translation, summarization, and question answering."
    )
}

sample_dir = "./chroma_sample_data"
os.makedirs(sample_dir, exist_ok=True)

for filename, content in docs.items():
    with open(os.path.join(sample_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

# load documents
loader = DirectoryLoader(sample_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
raw_documents = loader.load()
print(f"Loaded {len(raw_documents)} documents from: {sample_dir}")

---
## 2. تقطيع النصوص وتوليد التضمينات (Splitting & Embeddings)

In [ ]:
# 1 splitting the documents into smaller chunks for better processing
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
documents = splitter.split_documents(raw_documents)
print(f"Total chunks created: {len(documents)}")

# 2 embedding the documents using hugging face embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embedded_documents = embeddings.embed_documents([doc.page_content for doc in documents])
print(f"Generated embeddings for {len(embedded_documents)} chunks (vector dim: {len(embedded_documents[0])})")

---
## 3. الفهرسة والتخزين الدائم في ChromaDB (Persistence)

In [ ]:
# save the embeddings to a Chroma vector store
persist_directory = "./chroma_vector_store"
collection_name = "rag_mastery_collection"

# clean existing collection if present to prevent duplicate chunks on re-run
if os.path.exists(persist_directory):
    try:
        temp_db = Chroma(persist_directory=persist_directory, embedding_function=embeddings, collection_name=collection_name)
        temp_db.delete_collection()
    except Exception:
        pass
    shutil.rmtree(persist_directory, ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name=collection_name
)

print(f"save_directory := {persist_directory}")
print(f"Stored documents count: {vector_store._collection.count()}")

---
## 4. تحويل الفهرس إلى مسترجع (Retriever)

In [ ]:
# make the vector store retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"success in creating retriever: {retriever}")

---
## 5. تهيئة نموذج اللغة الكبير (LLM)

In [ ]:
# initialize the Groq chat model
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

print(f"LLM initialized successfully: {llm.model_name}")

---
## 6. تصميم قالب التوجيه (Prompt Template)

In [ ]:
# prompt template with strict grounding rules
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful AI assistant.

Answer the user's question using ONLY the information provided in the context.

Rules:
- Do not use outside knowledge.
- Do not make assumptions.
- Do not invent or hallucinate information.
- If the context does not contain enough information to answer the question,
  respond with "I don't know."
- Keep the answer accurate and concise.

Context:
{context}
"""
    ),
    (
        "human",
        "{input}"
    )
])

print("Prompt template created successfully!")

---
## 7. بناء سلسلة الـ RAG المتكاملة (`create_retrieval_chain`)

In [ ]:
# create document chain
document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)

# create full rag chain
rag_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=document_chain
)

print("success in creating the RAG chain")

---
## 8. اختبار وتشغيل مسار الـ RAG (Testing Queries)

In [ ]:
# test rag pipeline with user query
query = "What is deep learning and how does it differ from traditional machine learning?"

response = rag_chain.invoke({"input": query})

print(f"Question:\n{response['input']}\n")
print(f"Answer:\n{response['answer']}\n")
print("Retrieved Context Chunks:")
for idx, doc in enumerate(response["context"], start=1):
    src = os.path.basename(doc.metadata.get("source", ""))
    print(f"  [{idx}] ({src}): {doc.page_content[:90]}...")

---
## 9. تجربة أسئلة إضافية (Helper Function)

In [ ]:
def ask_rag(question: str):
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    res = rag_chain.invoke({"input": question})
    print(f"Answer:\n{res['answer']}")
    print("Sources:")
    for d in res["context"]:
        print(f"  - {os.path.basename(d.metadata.get('source', ''))}: {d.page_content[:70]}...")

# test questions from the knowledge base
ask_rag("What are CNNs best used for?")
ask_rag("What are the main types of machine learning?")

# test question outside the knowledge base (should say: I don't know.)
ask_rag("Who won the World Cup in 2022?")

## 🎯 ملخص مسار ChromaDB (المحاضرات 25، 26، 27):
1. قراءة المستندات بـ `DirectoryLoader` وتجزئتها بـ `RecursiveCharacterTextSplitter`.
2. توليد التضمينات محلياً وحفظها في `Chroma` على القرص (`persist_directory`).
3. تحويل الفهرس إلى `Retriever` يسترجع أفضل $k=3$ نتائج.
4. ربط الـ Retriever بـ `ChatGroq` عبر `create_retrieval_chain` للحصول على إجابات موثقة تمنع الهلوسة.

**المحاضرة التالية (28):** سنبني نفس المسار باستخدام لغة تعبير لانج تشين الحديثة (`LCEL`) بمشغل الـ `|` Pipeline!